# 01 - Consolidated Data Ingestion
This notebook handles the ingestion of ADS-B trajectory data (OpenSky), European flight schedules (Eurocontrol), and US flight schedules (BTS).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
import shutil
sys.path.append(os.path.abspath('..'))
import time
from datetime import datetime, timedelta
from src.utils import load_config
from src.data_ingestion import EurocontrolDownloader, BTSCombiner, EuroCombiner, BTSDownloader

config = load_config('../configs/config.yaml')

## Shared Date Window
Set the common start and end dates here so all ingestion sections target the same time range.

In [2]:
from pathlib import Path

INGEST_START_DATE = datetime(2022, 5, 1)
INGEST_END_DATE = datetime(2022, 5, 31)

if INGEST_END_DATE < INGEST_START_DATE:
    raise ValueError('INGEST_END_DATE must be on or after INGEST_START_DATE.')
if INGEST_START_DATE.year != INGEST_END_DATE.year:
    raise ValueError('This notebook currently expects the shared date window to stay within one calendar year.')

paths_cfg = config.get('paths', {})
DATA_DIR = (Path('..') / paths_cfg.get('data_dir', 'data')).resolve()
RAW_BASE_DIR = (Path('..') / paths_cfg.get('raw_data_dir', 'data/raw')).resolve()
PROCESSED_BASE_DIR = (Path('..') / paths_cfg.get('processed_data_dir', 'data/processed')).resolve()
OPENSKY_RAW_DIR = RAW_BASE_DIR / 'opensky'
OPENSKY_ARCHIVE_DIR = OPENSKY_RAW_DIR
EUROCONTROL_RAW_DIR = RAW_BASE_DIR / 'eurocontrol'
BTS_RAW_DIR = RAW_BASE_DIR / 'bts'

INGEST_YEAR = INGEST_START_DATE.year
INGEST_START_MONTH = INGEST_START_DATE.month
INGEST_END_MONTH = INGEST_END_DATE.month
INGEST_DATE_TAG = f"{INGEST_START_DATE:%Y%m%d}_{INGEST_END_DATE:%Y%m%d}"
INGEST_RANGE_LABEL = f"{INGEST_START_DATE:%Y-%m-%d} to {INGEST_END_DATE:%Y-%m-%d}"

def mondays_in_range(start_dt: datetime, end_dt: datetime) -> list[str]:
    current = start_dt
    while current.weekday() != 0:
        current += timedelta(days=1)
    values = []
    while current <= end_dt:
        values.append(current.strftime('%Y-%m-%d'))
        current += timedelta(days=7)
    return values

OPENSKY_SAMPLE_DATES = mondays_in_range(INGEST_START_DATE, INGEST_END_DATE)
ingest_cfg = config.get('ingestion', {})
ADSB_OUTPUT_FILE = str((Path('..') / ingest_cfg.get('adsb_combined_file', 'data/processed/adsb_combined.parquet')).resolve())
OPENSKY_SAMPLE_SINGLE_OUTPUT = ADSB_OUTPUT_FILE
EURO_OUTPUT_FILE = str((Path('..') / ingest_cfg.get('euro_combined_file', 'data/processed/eurocontrol_combined.parquet')).resolve())
BTS_OUTPUT_FILE = str((Path('..') / ingest_cfg.get('bts_combined_file', 'data/processed/bts_combined.parquet')).resolve())
CLEAN_DATASET_FILE = str(PROCESSED_BASE_DIR / f"clean_dataset_{INGEST_DATE_TAG}.parquet")

print('Shared range:', INGEST_RANGE_LABEL)
print('Data dir:', DATA_DIR)
print('Raw dir:', RAW_BASE_DIR)
print('Processed dir:', PROCESSED_BASE_DIR)
print('Raw source dirs:', {
    'opensky': OPENSKY_RAW_DIR,
    'eurocontrol': EUROCONTROL_RAW_DIR,
    'bts': BTS_RAW_DIR,
})
print('OpenSky archive dir:', OPENSKY_ARCHIVE_DIR)
print('OpenSky sample Mondays:', OPENSKY_SAMPLE_DATES)
print('ADSB output file:', ADSB_OUTPUT_FILE)
print('Euro output file:', EURO_OUTPUT_FILE)
print('BTS output file:', BTS_OUTPUT_FILE)
print('Clean dataset target:', CLEAN_DATASET_FILE)


Shared range: 2022-05-01 to 2022-05-31
Data dir: D:\Desertation\Code_v3\flight-disruption-prediction\data
Raw dir: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw
Processed dir: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed
Raw source dirs: {'opensky': WindowsPath('D:/Desertation/Code_v3/flight-disruption-prediction/data/raw/opensky'), 'eurocontrol': WindowsPath('D:/Desertation/Code_v3/flight-disruption-prediction/data/raw/eurocontrol'), 'bts': WindowsPath('D:/Desertation/Code_v3/flight-disruption-prediction/data/raw/bts')}
OpenSky archive dir: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw\opensky
OpenSky sample Mondays: ['2022-05-02', '2022-05-09', '2022-05-16', '2022-05-23', '2022-05-30']
ADSB output file: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\adsb_combined.parquet
Euro output file: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\eurocontrol_combined.parquet
BTS output file: D:\Desertatio

## Optional Reset
Use this cell to either clear the whole `data` directory or just `data/raw`, then recreate the expected `opensky`, `eurocontrol`, and `bts` raw source folders.


In [3]:
from pathlib import Path

RESET_ALL_DATA = False
RESET_RAW_DATA = False

removed_paths = []
if RESET_ALL_DATA:
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
        removed_paths.append(str(DATA_DIR))
    print(f'Reset data directory contents under: {DATA_DIR}')
elif RESET_RAW_DATA:
    if RAW_BASE_DIR.exists():
        shutil.rmtree(RAW_BASE_DIR)
        removed_paths.append(str(RAW_BASE_DIR))
    print(f'Reset raw directory contents under: {RAW_BASE_DIR}')
else:
    print('No reset selected. Ensuring expected data directories exist.')

for path in [DATA_DIR, RAW_BASE_DIR, PROCESSED_BASE_DIR, OPENSKY_RAW_DIR, EUROCONTROL_RAW_DIR, BTS_RAW_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if removed_paths:
    print('Removed paths:')
    for path_str in removed_paths:
        print(' -', path_str)

print('Ready data layout:')
for name, path in {
    'data': DATA_DIR,
    'raw': RAW_BASE_DIR,
    'processed': PROCESSED_BASE_DIR,
    'opensky': OPENSKY_RAW_DIR,
    'eurocontrol': EUROCONTROL_RAW_DIR,
    'bts': BTS_RAW_DIR,
}.items():
    print(f' - {name}: {path}')


No reset selected. Ensuring expected data directories exist.
Ready data layout:
 - data: D:\Desertation\Code_v3\flight-disruption-prediction\data
 - raw: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw
 - processed: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed
 - opensky: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw\opensky
 - eurocontrol: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw\eurocontrol
 - bts: D:\Desertation\Code_v3\flight-disruption-prediction\data\raw\bts


## 1. OpenSky ADS-B Ingestion
Use the public OpenSky weekly sample bucket to build the ADS-B source dataset for the shared date window.

### Optional: OpenSky Weekly Sample Bucket
OpenSky publishes public Monday sample datasets with hourly state vectors. This runner derives its sample dates from the shared date window above.

In [4]:
import subprocess
import sys
from pathlib import Path

OPENSKY_SAMPLE_HOUR_START = 0
OPENSKY_SAMPLE_HOUR_END = 23
OPENSKY_SAMPLE_MAX_WORKERS = 4
OPENSKY_SAMPLE_OVERWRITE = False
OPENSKY_SAMPLE_URL_TEMPLATE = (
    'https://s3.opensky-network.org/data-samples/states/{date}/{hour:02d}/'
    'states_{date}-{hour:02d}.avro.tar'
)

sample_script_path = Path('../scripts/fetch_opensky_samples.py').resolve()
sample_command = [
    sys.executable,
    str(sample_script_path),
    '--hour-start', str(OPENSKY_SAMPLE_HOUR_START),
    '--hour-end', str(OPENSKY_SAMPLE_HOUR_END),
    '--max-workers', str(OPENSKY_SAMPLE_MAX_WORKERS),
    '--archive-dir', str(OPENSKY_ARCHIVE_DIR),
    '--url-template', OPENSKY_SAMPLE_URL_TEMPLATE,
]
if OPENSKY_SAMPLE_SINGLE_OUTPUT:
    sample_command.extend(['--single-output-file', OPENSKY_SAMPLE_SINGLE_OUTPUT])


for sample_date in OPENSKY_SAMPLE_DATES:
    sample_command.extend(['--sample-date', sample_date])

if OPENSKY_SAMPLE_OVERWRITE:
    sample_command.append('--overwrite')

print('Running:', ' '.join(sample_command))
process = subprocess.Popen(
    sample_command,
    cwd=Path('..').resolve(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
)
if process.stdout is not None:
    for line in process.stdout:
        print(line, end='')
completed_returncode = process.wait()
process.stdout.close() if process.stdout is not None else None
if completed_returncode != 0:
    raise RuntimeError(f'OpenSky sample script failed with exit code {completed_returncode}')


Running: D:\Desertation\Code_v3\.venv11\Scripts\python.exe D:\Desertation\Code_v3\flight-disruption-prediction\scripts\fetch_opensky_samples.py --hour-start 0 --hour-end 23 --max-workers 4 --archive-dir D:\Desertation\Code_v3\flight-disruption-prediction\data\raw\opensky --url-template https://s3.opensky-network.org/data-samples/states/{date}/{hour:02d}/states_{date}-{hour:02d}.avro.tar --single-output-file D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\adsb_combined.parquet --sample-date 2022-05-02 --sample-date 2022-05-09 --sample-date 2022-05-16 --sample-date 2022-05-23 --sample-date 2022-05-30


INFO:fetch_opensky_samples:Combined parquet already exists, skipping: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\adsb_combined.parquet


## 2. Eurocontrol (OPDI) Schedule Ingestion
Download the monthly Parquet flight lists for the shared date window.

In [5]:
euro = EurocontrolDownloader()
year = INGEST_YEAR

for month in range(INGEST_START_MONTH, INGEST_END_MONTH + 1):
    euro.download_month(year, month, RAW_BASE_DIR)


## 3. BTS (US) Schedule Ingestion
Best-effort BTS automation using the TranStats `PREZIP` monthly archive pattern, followed by combination into a single parquet file for the shared date window.

In [6]:
bts = BTSDownloader()
for month in range(INGEST_START_MONTH, INGEST_END_MONTH + 1):
    bts.download_month(INGEST_YEAR, month, RAW_BASE_DIR)

combiner = BTSCombiner()
combiner.combine_csvs(BTS_RAW_DIR, BTS_OUTPUT_FILE)


Unknown IATA airport code 'TYS' - using fallback 'KTYS'


Unknown IATA airport code 'FNT' - using fallback 'KFNT'


Unknown IATA airport code 'PWM' - using fallback 'KPWM'


Unknown IATA airport code 'MHT' - using fallback 'KMHT'


Unknown IATA airport code 'ECP' - using fallback 'KECP'


...

Unknown IATA airport code 'AKN' - using fallback 'KAKN'


Unknown IATA airport code 'ADK' - using fallback 'KADK'


Unknown IATA airport code 'GST' - using fallback 'KGST'


Unknown IATA airport code 'HYA' - using fallback 'KHYA'


WindowsPath('D:/Desertation/Code_v3/flight-disruption-prediction/data/processed/bts_combined.parquet')

## 4. Build Source Datasets
After ingestion, this cell gives you the three source datasets we use downstream: ADS-B, Eurocontrol, and BTS.

In [7]:
legacy_output_map = {
    PROCESSED_BASE_DIR / f'euro_dataset_{INGEST_DATE_TAG}.parquet': Path(EURO_OUTPUT_FILE),
    PROCESSED_BASE_DIR / f'bts_dataset_{INGEST_DATE_TAG}.parquet': Path(BTS_OUTPUT_FILE),
    PROCESSED_BASE_DIR / f'adsb_dataset_{INGEST_DATE_TAG}.parquet': Path(ADSB_OUTPUT_FILE),
    PROCESSED_BASE_DIR / f'states_{INGEST_DATE_TAG}.parquet': Path(ADSB_OUTPUT_FILE),
    OPENSKY_RAW_DIR / f'states_{INGEST_DATE_TAG}.parquet': Path(ADSB_OUTPUT_FILE),
}

for legacy_path, combined_path in legacy_output_map.items():
    if legacy_path.exists() and not combined_path.exists():
        legacy_path.replace(combined_path)
        print(f'Renamed legacy output: {legacy_path.name} -> {combined_path.name}')

if not Path(EURO_OUTPUT_FILE).exists():
    euro_combiner = EuroCombiner()
    euro_combiner.combine_parquets(EUROCONTROL_RAW_DIR, EURO_OUTPUT_FILE)

source_datasets = {
    'adsb': Path(ADSB_OUTPUT_FILE),
    'euro': Path(EURO_OUTPUT_FILE),
    'bts': Path(BTS_OUTPUT_FILE),
}

for name, dataset_path in source_datasets.items():
    if dataset_path.exists():
        print(f'{name}: {dataset_path} ({dataset_path.stat().st_size:,} bytes)')
    else:
        print(f'{name}: missing -> {dataset_path}')

print(f'Next target clean dataset path: {CLEAN_DATASET_FILE}')


adsb: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\adsb_combined.parquet (10,345,382,181 bytes)
euro: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\eurocontrol_combined.parquet (28,469,037 bytes)
bts: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\bts_combined.parquet (13,880,320 bytes)
Next target clean dataset path: D:\Desertation\Code_v3\flight-disruption-prediction\data\processed\clean_dataset_20220501_20220531.parquet
